# Version 5 — critical points with $e\neq0$: positions, for the three groups

One method, applied group by group. Everything is reported with its residual, and the positions
are collected in a table at the end.

**Setting.** $\Phi$ is the Section-6 functional. In the eccentric anomaly the singular weight
cancels ($W_e\,d\theta=\tfrac12 dE$), so
$$\Phi=\tfrac12\int_0^{2\pi}S(\hat u(E))\,dE,\qquad
\hat u(E)=\frac{(\cos E-e)p+\sqrt{1-e^2}\sin E\,q}{\|\cdot\|},$$
with $p$ the pericentre direction, $q=\nu\times p$, $\nu$ the orbit-plane normal.

**Method, in the order executed.**
1. *Disc, exactly.* At fixed plane, $\Phi=\pi[a_0+\operatorname{Re}G(z)]$ with
   $z=\lambda e^{i\psi_3}$, $\lambda=\frac{1-\sqrt{1-e^2}}{e}$. The disc critical points are the
   zeros of $G'$, and $G'(z)=zP(z^2)$ — a polynomial, solved by one companion matrix. Each root is
   validated against the quadrature, since truncation throws spurious roots near $|z|=1$.
2. *Boundary of the fundamental domain*, by symmetric criticality, solved by **bisection**.
3. *Interior*, seeded by the exact disc solve and closed by a $4$-D Newton.

**Accuracy control.** The integrand peaks with width equal to the distance from the great circle to
the nearest rotation axis, so the quadrature resolution is tied to that distance,
$N\approx200/\mathrm{gap}$. Fixed $N$ silently produces false critical points near $e\to1$.
Every point is finally verified at $N\ge32768$ and must satisfy $\|\nabla\Phi\|<10^{-5}$.

In [1]:
using LinearAlgebra, Printf

function rot(n, α)
    n = normalize(float.(collect(n)))
    K = [0 -n[3] n[2]; n[3] 0 -n[1]; -n[2] n[1] 0]
    Matrix(1.0I,3,3) + sin(α)*K + (1-cos(α))*K^2
end
function closure(gens)
    G = [Matrix(1.0I,3,3)]; ch = true
    while ch
        ch = false
        for X in copy(G), s in gens
            Y = X*s; any(M -> norm(M-Y) < 1e-9, G) || (push!(G,Y); ch = true)
        end
    end
    G
end
const φg  = (1+sqrt(5))/2
const MIN = -Matrix(1.0I,3,3)
## these generate exactly the same groups as the paper's ⟨A,B⟩ (checked matrix by matrix)
Tet  = closure([rot([1,0,0], pi),   rot([1,1,1], 2pi/3)])
Oct  = closure([rot([0,0,1], pi/2), rot([1,1,1], 2pi/3)])
Ico  = closure([rot([0,0,1], pi),   rot([0,1,φg], 2pi/5)])
GamC = closure([rot([0,0,1], pi/2), rot([1,1,1], 2pi/3), MIN])
GamI = closure([rot([0,0,1], pi),   rot([0,1,φg], 2pi/5), MIN])
aw(G) = [(vec(nullspace(L-Matrix(1.0I,3,3))), sqrt(max(3-tr(L),0.0)))
         for L in G if norm(L-Matrix(1.0I,3,3)) > 1e-9]

const J1 = [0.0 0 0; 0 0 -1; 0 1 0]
const J2 = [0.0 0 -1; 0 0 0; 1 0 0]     # the paper's J₂ = minus the standard generator about +y
const J3 = [0.0 -1 0; 1 0 0; 0 0 0]
Rpsi(a,b,c) = exp(a*J1)*exp(b*J2)*exp(c*J3)
efromlam(l) = 2l/(1+l^2)
Nfor(g) = clamp(2^ceil(Int, log2(200/max(g,0.01))), 2048, 32768)

function PhiG(e, p, q, nax, cax; N = 4096)
    pa = [dot(p,n) for n in nax]; pb = [dot(q,n) for n in nax]
    ce = sqrt(max(1-e^2, 0.0)); acc = 0.0
    for k in 0:N-1
        E = 2pi*k/N
        x = cos(E) - e; y = ce*sin(E); r = sqrt(x*x + y*y)
        cx = x/r; cy = y/r; S = 0.0
        for j in eachindex(nax)
            t = cx*pa[j] + cy*pb[j]
            S += 1/(cax[j]*sqrt(max(1 - t*t, 1e-300)))
        end
        acc += S
    end
    0.5*acc*2pi/N
end
Phi4(x, nax, cax; N=4096) =
    (R = Rpsi(x[2],x[3],x[4]); PhiG(efromlam(x[1]), R[:,1], R[:,2], nax, cax; N=N))

fd_t(G,p0) = hcat([M*p0 for M in G]...)
inFD(ν,p0,P) = dot(ν,p0) ≥ maximum(P'*ν) - 1e-12
function fd_vertices(Gam, p0)
    P = fd_t(Gam,p0); V = Vector{Float64}[]
    for L in Gam
        (det(L) > 0.5 && norm(L-Matrix(1.0I,3,3)) > 1e-9) || continue
        n = vec(nullspace(L-Matrix(1.0I,3,3)))
        for s in (n,-n); inFD(s,p0,P) && !any(w->norm(w-s)<1e-8,V) && push!(V,s); end
    end
    V
end
refl(m) = Matrix(1.0I,3,3) - 2*m*m'
isin(M,G) = any(X -> norm(X-M) < 1e-8, G)
nuof(φ,θ) = [sin(θ)*cos(φ), sin(θ)*sin(φ), cos(θ)]
psi12(ν)  = (atan(-ν[2], ν[3]), -asin(clamp(ν[1],-1,1)))
println("groups: |T|=", length(Tet), " |O|=", length(Oct), " |I|=", length(Ico),
        "   |Γ_cub|=", length(GamC), " |Γ_ico|=", length(GamI))

groups: |T|=12 |O|=24 |I|=60   |Γ_cub|=48 |Γ_ico|=120


## Step 1 — the exact disc solve, with validation

In [2]:
function harmonics(ψ1, ψ2, nax, cax, K, N)
    R0 = exp(ψ1*J1)*exp(ψ2*J2); a = R0[:,1]; b = R0[:,2]
    pa = [dot(a,n) for n in nax]; pb = [dot(b,n) for n in nax]
    s = Vector{Float64}(undef, N)
    for k in 0:N-1
        α = 2pi*k/N; c = cos(α); sn = sin(α); S = 0.0
        for j in eachindex(nax)
            t = c*pa[j] + sn*pb[j]
            S += 1/(cax[j]*sqrt(max(1 - t*t, 1e-300)))
        end
        s[k+1] = S
    end
    ak = zeros(K+1); bk = zeros(K+1)
    for m in 2:2:K
        ca = 0.0; cb = 0.0
        for k in 0:N-1
            α = 2pi*k/N; ca += s[k+1]*cos(m*α); cb += s[k+1]*sin(m*α)
        end
        ak[m+1] = 2ca/N; bk[m+1] = 2cb/N
    end
    ak, bk
end
function disc_roots(ψ1, ψ2, nax, cax, K, N)
    ak, bk = harmonics(ψ1, ψ2, nax, cax, K, N)
    p = ComplexF64[]
    for m in 1:(K ÷ 2); k = 2m; push!(p, k*(ak[k+1] - im*bk[k+1])); end
    mx = maximum(abs, p)
    while length(p) > 1 && abs(p[end]) < 1e-13*mx; pop!(p); end
    length(p) < 2 && return ComplexF64[]
    n = length(p)-1
    C = zeros(ComplexF64,n,n)
    for i in 1:n-1; C[i+1,i] = 1.0; end
    for i in 1:n; C[i,n] = -p[i]/p[end]; end
    zs = ComplexF64[]
    for w in eigvals(C)
        abs(w) < 0.92 || continue
        for z in (sqrt(w), -sqrt(w)); 0.03 < abs(z) < 0.955 && push!(zs, z); end
    end
    zs
end
function gradpart(x, idx, nax, cax, N; h = 1e-5)
    g = zeros(length(idx))
    for (k,i) in enumerate(idx)
        d = zeros(4); d[i] = h
        g[k] = (Phi4(x+d,nax,cax;N=N) - Phi4(x-d,nax,cax;N=N))/(2h)
    end
    g
end
grad4(x, nax, cax; N=8192) = gradpart(x, 1:4, nax, cax, N)
function hess4(x, nax, cax; h=2e-3, N=8192)
    H = zeros(4,4)
    for i in 1:4
        d = zeros(4); d[i] = h
        H[i,i] = (Phi4(x+d,nax,cax;N=N) - 2Phi4(x,nax,cax;N=N) + Phi4(x-d,nax,cax;N=N))/h^2
        for j in i+1:4
            u = zeros(4); u[i]=h; v = zeros(4); v[j]=h
            H[i,j] = H[j,i] = (Phi4(x+u+v,nax,cax;N=N) - Phi4(x+u-v,nax,cax;N=N)
                             - Phi4(x-u+v,nax,cax;N=N) + Phi4(x-u-v,nax,cax;N=N))/(4h^2)
        end
    end
    H
end
function newton4(x0, nax, cax; N = 8192, iters = 60)
    x = copy(x0); g = grad4(x,nax,cax;N=N); ng = norm(g)
    for _ in 1:iters
        d = try hess4(x,nax,cax;N=N) \ g catch; return nothing end
        all(isfinite,d) || return nothing
        nd = norm(d); nd > 0.1 && (d *= 0.1/nd)
        t = 1.0; ok = false
        for _ in 1:25
            y = x - t*d; y[1] = clamp(y[1], 0.03, 0.955)
            gy = grad4(y,nax,cax;N=N); ngy = norm(gy)
            if ngy < ng; x = y; g = gy; ng = ngy; ok = true; break end
            t /= 2
        end
        ok || break
        ng < 1e-9 && break
    end
    ng < 1e-6 ? x : nothing
end

newton4 (generic function with 1 method)

## Step 2 — the boundary of the fundamental domain (bisection)

In [3]:
function edge_search(g2, gapf, tmax)
    ts = collect(range(1e-3, tmax-1e-3, length = 500))
    okt(t) = gapf(t) > 0.03
    function roots_t(l)
        out = Float64[]; gl(t) = g2(l,t)[1]
        prev = nothing; prevt = 0.0
        for t in ts
            if !okt(t); prev = nothing; continue; end
            v = gl(t)
            if prev !== nothing && prev*v < 0
                a0 = prevt; b0 = t; ga = prev
                for _ in 1:60
                    m0 = (a0+b0)/2; gm = gl(m0)
                    if (gm > 0) == (ga > 0); a0 = m0; ga = gm else b0 = m0 end
                end
                push!(out, (a0+b0)/2)
            end
            prev = v; prevt = t
        end
        out
    end
    sols = Vector{Float64}[]; prevroots = Tuple{Float64,Float64}[]; prevλ = 0.0
    for l in range(0.03, 0.95, length = 96)
        cur = [(t, g2(l,t)[2]) for t in roots_t(l)]
        for (t,gt) in cur, (tp,gtp) in prevroots
            (abs(t-tp) < 0.08 && gt*gtp < 0) || continue
            a0 = prevλ; b0 = l; ga = gtp
            for _ in 1:45
                m0 = (a0+b0)/2; rm = roots_t(m0); isempty(rm) && break
                gm = g2(m0, rm[argmin(abs.(rm .- t))])[2]
                if (gm > 0) == (ga > 0); a0 = m0; ga = gm else b0 = m0 end
            end
            lst = (a0+b0)/2; rr = roots_t(lst); isempty(rr) && continue
            v = [lst, rr[argmin(abs.(rr .- t))]]
            (norm(g2(v...)) < 1e-4 && v[1] > 0.03 && gapf(v[2]) > 0.03) || continue
            any(w -> norm(w-v) < 1e-4, sols) || push!(sols, v)
        end
        prevroots = cur; prevλ = l
    end
    sols
end
function boundary_pass(H, Gam, p0)
    AW = aw(H); nax = [n for (n,c) in AW]; cax = [c for (n,c) in AW]
    gap(ν) = minimum(abs(dot(ν,n)) for n in nax)
    V = fd_vertices(Gam, p0); found = Vector{Float64}[]
    for (i,j) in ((1,2),(1,3),(2,3))
        v1 = V[i]; v2 = V[j]; m = normalize(cross(v1,v2))
        isin(refl(m), Gam) || continue
        tmax = acos(clamp(dot(v1,v2),-1,1)); w2 = normalize(v2 - dot(v2,v1)*v1)
        arc(t) = normalize(v1*cos(t) + w2*sin(t))
        for pf in (ν -> normalize(cross(ν,m)), ν -> m)
            F(l,t) = (ν = arc(t); p = pf(ν); PhiG(efromlam(l), p, cross(ν,p), nax, cax; N = Nfor(gap(ν))))
            g2(l,t; h=1e-5) = [(F(l+h,t)-F(l-h,t))/(2h), (F(l,t+h)-F(l,t-h))/(2h)]
            for v in edge_search(g2, t -> gap(arc(t)), tmax)
                ν = arc(v[2]); p = pf(ν)
                ψ1, ψ2 = psi12(ν); R0 = exp(ψ1*J1)*exp(ψ2*J2)
                push!(found, [v[1], ψ1, ψ2, atan(dot(p,R0[:,2]), dot(p,R0[:,1]))])
            end
        end
    end
    found
end

boundary_pass (generic function with 1 method)

## Step 3 — the interior, and the group driver

In [4]:
function interior_pass(H, Gam, p0; nφ, nθ, wall = 0.03, K = 48, nseed = 45)
    AW = aw(H); nax = [n for (n,c) in AW]; cax = [c for (n,c) in AW]
    P = fd_t(Gam, p0)
    gapψ(a,b) = (ν = Rpsi(a,b,0.0)[:,3]; minimum(abs(dot(ν,n)) for n in nax))
    seeds = Tuple{Float64,Vector{Float64}}[]; npl = 0; nrt = 0
    for i in 0:nφ-1, j in 1:nθ-1
        ν = nuof(2pi*i/nφ, pi*j/nθ)
        inFD(ν, p0, P) || continue
        ψ1, ψ2 = psi12(ν); g = gapψ(ψ1,ψ2); g < wall && continue
        npl += 1; Nq = Nfor(g)
        for z in disc_roots(ψ1, ψ2, nax, cax, K, Nq)
            x = [abs(z), ψ1, ψ2, angle(z)]
            norm(gradpart(x, [1,4], nax, cax, Nq)) < 1e-2 || continue
            nrt += 1
            push!(seeds, (norm(gradpart(x, [2,3], nax, cax, Nq)), x))
        end
    end
    sort!(seeds, by = t -> t[1])
    starts = Vector{Float64}[]
    for (_, x) in seeds
        any(y -> norm(y-x) < 0.05, starts) && continue
        push!(starts, x); length(starts) ≥ nseed && break
    end
    out = Vector{Float64}[]
    for x0 in starts
        x = newton4(x0, nax, cax)
        x === nothing && continue
        (x[1] > 0.03 && x[1] < 0.95 && gapψ(x[2],x[3]) > 0.02) && push!(out, x)
    end
    out, npl, nrt, (isempty(seeds) ? NaN : seeds[1][1]), length(starts)
end

function same(x, y, Gam)
    abs(x[1]-y[1]) > 1e-4 && return false
    R1 = Rpsi(x[2],x[3],x[4]); R2 = Rpsi(y[2],y[3],y[4])
    n1 = R1[:,3]; p1 = R1[:,1]; n2 = R2[:,3]; p2 = R2[:,1]
    for γ in Gam
        n = det(γ)*(γ*n1); p = γ*p1
        (norm(n-n2) < 1e-4 && (norm(p-p2) < 1e-4 || norm(p+p2) < 1e-4)) && return true
    end
    false
end

function report(name, H, Gam, p0; nφ, nθ)
    AW = aw(H); nax = [n for (n,c) in AW]; cax = [c for (n,c) in AW]
    gapν(ν) = minimum(abs(dot(ν,n)) for n in nax)
    println("\n", "="^96); println(name); println("="^96)
    t0 = time(); bnd = boundary_pass(H, Gam, p0)
    @printf("step 2  boundary of D : %d point(s)   [%.0f s]\n", length(bnd), time()-t0)
    t1 = time(); int, npl, nrt, best, nst = interior_pass(H, Gam, p0; nφ=nφ, nθ=nθ)
    @printf("step 3  interior of D : %d planes, %d validated disc roots, %d seeds,\n", npl, nrt, nst)
    @printf("                        smallest tilt residual on the grid %.4f,  Newton gave %d   [%.0f s]\n",
            best, length(int), time()-t1)
    ver = Vector{Float64}[]; nrej = 0
    for x in vcat(bnd, int)
        R = Rpsi(x[2],x[3],x[4]); Nv = max(32768, Nfor(gapν(R[:,3])))
        if norm(grad4(x, nax, cax; N = Nv)) < 1e-5; push!(ver, x) else nrej += 1 end
    end
    @printf("verification at N ≥ 32768 : %d kept, %d rejected\n", length(ver), nrej)
    all = Vector{Float64}[]
    for x in ver; any(y -> same(x,y,Gam), all) || push!(all, x); end
    sort!(all, by = x -> Phi4(x, nax, cax; N = 8192))
    println("-"^96)
    @printf("%s :  %d INEQUIVALENT CRITICAL POINT(S) WITH e > 0\n", name, length(all))
    println("-"^96)
    for (k,x) in enumerate(all)
        R = Rpsi(x[2],x[3],x[4]); ν = R[:,3]; p = R[:,1]
        Nv = max(32768, Nfor(gapν(ν))); ev = eigvals(hess4(x, nax, cax; N = Nv))
        @printf("\n #%d   POSITION\n", k)
        @printf("      e    = %.12f            (λ = %.12f)\n", efromlam(x[1]), x[1])
        @printf("      ψ₁   = %+.9f°\n      ψ₂   = %+.9f°\n      ψ₃   = %+.9f°\n", rad2deg.(x[2:4])...)
        @printf("      ν    = (%+.12f, %+.12f, %+.12f)     [orbit-plane normal]\n", ν...)
        @printf("      peri = (%+.12f, %+.12f, %+.12f)     [pericentre direction]\n", p...)
        @printf("      Φ    = %.12f\n", Phi4(x, nax, cax; N = Nv))
        @printf("      ‖∇Φ‖ = %.2e  (N = %d)     distance to collision set = %.5f\n",
                norm(grad4(x, nax, cax; N = Nv)), Nv, gapν(ν))
        @printf("      eig ∇²Φ = (%+.4f, %+.4f, %+.4f, %+.4f)\n", ev...)
        @printf("      det ∇²Φ = %+.5e    signature (%d,%d)   %s\n",
                prod(ev), count(>(0),ev), count(<(0),ev),
                abs(prod(ev)) > 1e-4 ? "NONDEGENERATE" : "DEGENERATE")
    end
    [(name, x, nax, cax) for x in all]
end

report (generic function with 1 method)

## The tetrahedral group

In [5]:
res_T = report("T   tetrahedral,  n = 12", Tet, GamC, normalize([3.0,2.0,1.0]); nφ = 380, nθ = 190)


T   tetrahedral,  n = 12
step 2  boundary of D : 1 point(s)   [52 s]
step 3  interior of D : 805 planes, 2250 validated disc roots, 45 seeds,
                        smallest tilt residual on the grid 0.3527,  Newton gave 45   [14 s]
verification at N ≥ 32768 : 46 kept, 0 rejected
------------------------------------------------------------------------------------------------
T   tetrahedral,  n = 12 :  1 INEQUIVALENT CRITICAL POINT(S) WITH e > 0
------------------------------------------------------------------------------------------------

 #1   POSITION
      e    = 0.844835651500            (λ = 0.550372242735)
      ψ₁   = -45.000000000°
      ψ₂   = -76.550274487°
      ψ₃   = +180.000000000°
      ν    = (+0.972574383924, +0.164467424943, +0.164467424943)     [orbit-plane normal]
      peri = (-0.232592062922, +0.687713942081, +0.687713942081)     [pericentre direction]
      Φ    = 27.329792082829
      ‖∇Φ‖ = 1.42e-08  (N = 32768)     distance to collision set = 0.16447
    

1-element Vector{Tuple{String, Vector{Float64}, Vector{Vector{Float64}}, Vector{Float64}}}:
 ("T   tetrahedral,  n = 12", [0.550372242734871, -0.7853981633974483, -1.3360543331113983, 3.141592653589793], [[1.0, 0.0, 0.0], [-0.5773502691896258, -0.5773502691896258, -0.5773502691896257], [0.5773502691896257, -0.5773502691896255, 0.5773502691896256], [-0.5773502691896257, -0.5773502691896255, 0.5773502691896256], [-0.5773502691896257, -0.5773502691896257, -0.5773502691896257], [0.5773502691896258, -0.5773502691896258, -0.5773502691896257], [-0.5773502691896261, -0.5773502691896257, 0.5773502691896256], [-0.5773502691896257, 0.5773502691896255, 0.5773502691896256], [0.5773502691896261, -0.5773502691896257, 0.5773502691896256], [0.0, -0.9999999999999999, 1.0532500405730103e-16], [0.0, -1.1102230246251565e-16, 1.0]], [2.0, 1.7320508075688776, 1.7320508075688772, 1.7320508075688772, 1.7320508075688772, 1.7320508075688776, 1.7320508075688772, 1.7320508075688772, 1.7320508075688772, 2.0, 2.0])

## The octahedral group

In [6]:
res_O = report("O   octahedral,   n = 24", Oct, GamC, normalize([3.0,2.0,1.0]); nφ = 380, nθ = 190)


O   octahedral,   n = 24
step 2  boundary of D : 0 point(s)   [0 s]
step 3  interior of D : 654 planes, 2390 validated disc roots, 45 seeds,
                        smallest tilt residual on the grid 0.6165,  Newton gave 31   [34 s]
verification at N ≥ 32768 : 31 kept, 0 rejected
------------------------------------------------------------------------------------------------
O   octahedral,   n = 24 :  2 INEQUIVALENT CRITICAL POINT(S) WITH e > 0
------------------------------------------------------------------------------------------------

 #1   POSITION
      e    = 0.873524145025            (λ = 0.587527181187)
      ψ₁   = -53.539710624°
      ψ₂   = -62.034213207°
      ψ₃   = +54.935586296°
      ν    = (+0.883227772443, +0.377157282686, +0.278677387140)     [orbit-plane normal]
      peri = (+0.269407055426, +0.078314833365, -0.959836770165)     [pericentre direction]
      Φ    = 62.157462618621
      ‖∇Φ‖ = 5.14e-08  (N = 32768)     distance to collision set = 0.06964
      

2-element Vector{Tuple{String, Vector{Float64}, Vector{Vector{Float64}}, Vector{Float64}}}:
 ("O   octahedral,   n = 24", [0.5875271811873533, -0.9344442309479292, -1.0827012693496127, 0.9588068573833824], [[0.0, 0.0, 1.0], [-0.5773502691896258, -0.5773502691896258, -0.5773502691896257], [0.0, 0.0, 1.0], [-0.0, 0.7071067811865475, 0.7071067811865475], [-0.7071067811865475, 0.0, -0.7071067811865475], [-0.5773502691896257, -0.5773502691896257, -0.5773502691896257], [0.0, 0.0, 1.0], [-0.5773502691896257, 0.577350269189626, 0.5773502691896256], [-0.0, 0.9999999999999999, -3.322193591245671e-16], [0.577350269189626, -0.5773502691896258, 0.5773502691896257]  …  [-0.5773502691896256, -0.5773502691896258, 0.577350269189626], [-0.0, 1.0, -7.183105657178396e-17], [-0.5773502691896256, -0.5773502691896257, 0.5773502691896256], [-0.577350269189626, 0.5773502691896255, 0.5773502691896258], [-0.0, 0.7071067811865475, -0.7071067811865476], [0.7071067811865474, 1.5700924586837757e-16, -0.7071067811865

## The icosahedral group

Its fundamental domain is $1/120$ of the sphere and the collision set has $31$ great circles, so the
grid must be much finer here.

In [7]:
res_I = report("I   icosahedral,  n = 60", Ico, GamI, normalize([0.17,0.41,1.0]); nφ = 900, nθ = 450)


I   icosahedral,  n = 60
step 2  boundary of D : 0 point(s)   [0 s]
step 3  interior of D : 1536 planes, 6220 validated disc roots, 45 seeds,
                        smallest tilt residual on the grid 3.1532,  Newton gave 24   [80 s]
verification at N ≥ 32768 : 24 kept, 0 rejected
------------------------------------------------------------------------------------------------
I   icosahedral,  n = 60 :  1 INEQUIVALENT CRITICAL POINT(S) WITH e > 0
------------------------------------------------------------------------------------------------

 #1   POSITION
      e    = 0.679505708740            (λ = 0.391946346474)
      ψ₁   = -28.072163853°
      ψ₂   = -5.295686121°
      ψ₃   = -181.342356688°
      ν    = (+0.092295617693, +0.468574645800, +0.878589392304)     [orbit-plane normal]
      peri = (-0.995458385678, +0.064091268850, +0.070391133258)     [pericentre direction]
      Φ    = 168.310940219575
      ‖∇Φ‖ = 8.08e-08  (N = 32768)     distance to collision set = 0.03169
    

1-element Vector{Tuple{String, Vector{Float64}, Vector{Vector{Float64}}, Vector{Float64}}}:
 ("I   icosahedral,  n = 60", [0.39194634647403015, -0.48995168739477696, -0.09242715897008405, -3.165021197532224], [[0.0, 0.0, 1.0], [-0.0, 0.5257311121191333, 0.85065080835204], [0.35682208977308977, -1.4616672547270628e-17, -0.9341723589627157], [-0.35682208977308977, 1.4616672547270628e-17, -0.9341723589627157], [0.0, 0.5257311121191335, 0.8506508083520399], [-0.0, -0.5257311121191333, 0.85065080835204], [-0.8506508083520401, 1.1102230246251565e-16, 0.5257311121191336], [-0.0, 0.5257311121191336, -0.8506508083520399], [-0.8506508083520401, 1.1102230246251565e-16, -0.5257311121191336], [-0.0, 0.5257311121191336, 0.85065080835204]  …  [0.5257311121191336, 0.8506508083520399, 1.633609243278222e-17], [-0.5257311121191335, 0.85065080835204, -1.4795577209475128e-17], [0.8090169943749476, -0.5, -0.30901699437494745], [-0.8090169943749473, -0.5000000000000002, 0.30901699437494756], [-0.809016994374

## Positions — summary

In [8]:
println("\n", "#"^96)
println("CRITICAL POINTS OF Φ WITH 0 < e < 1  (positions, up to the symmetry Γ)")
println("#"^96)
for (nm, x, nax, cax) in vcat(res_T, res_O, res_I)
    R = Rpsi(x[2],x[3],x[4]); ν = R[:,3]; p = R[:,1]
    @printf("\n%s\n", nm)
    @printf("   e  = %.12f\n", efromlam(x[1]))
    @printf("   ψ  = (%+.9f°, %+.9f°, %+.9f°)\n", rad2deg.(x[2:4])...)
    @printf("   ν  = (%+.12f, %+.12f, %+.12f)\n", ν...)
    @printf("   p  = (%+.12f, %+.12f, %+.12f)\n", p...)
    @printf("   Φ  = %.12f      ‖∇Φ‖ = %.1e\n",
            Phi4(x, nax, cax; N = 32768), norm(grad4(x, nax, cax; N = 32768)))
end
@printf("\ncounts:   T : %d      O : %d      I : %d\n", length(res_T), length(res_O), length(res_I))


################################################################################################
CRITICAL POINTS OF Φ WITH 0 < e < 1  (positions, up to the symmetry Γ)
################################################################################################

T   tetrahedral,  n = 12
   e  = 0.844835651500
   ψ  = (-45.000000000°, -76.550274487°, +180.000000000°)
   ν  = (+0.972574383924, +0.164467424943, +0.164467424943)
   p  = (-0.232592062922, +0.687713942081, +0.687713942081)
   Φ  = 27.329792082829      ‖∇Φ‖ = 1.4e-08

O   octahedral,   n = 24
   e  = 0.873524145025
   ψ  = (-53.539710624°, -62.034213207°, +54.935586296°)
   ν  = (+0.883227772443, +0.377157282686, +0.278677387140)
   p  = (+0.269407055426, +0.078314833365, -0.959836770165)
   Φ  = 62.157462618621      ‖∇Φ‖ = 5.1e-08

O   octahedral,   n = 24
   e  = 0.962477021339
   ψ  = (-48.226440874°, -42.846727545°, -11.005530606°)
   ν  = (+0.680039471268, +0.546790201125, +0.488432997935)
   p  = (+0.719691496346, -